# DyNECT GNE debugging — bottom-up reconstruction

**Goal:** Reproduce the successful 2-prosumer battery/P2P toy, then change **one complexity dimension at a time** before bringing back the full energy model. This is a diagnostic experiment, **not yet a full research model**.

The ladder uses the real `DynLQGameTV → MpAVI.from_game → StaticGNEP → MpAVI.nabetani → GNESetSolver` interfaces. Every stage uses a *new subprocess*, a **timeout only** (no RAM watchdog), and an unmodified Julia solver. It keeps the Nabetani β parameterization; there are no unsafe Julia monkey-patches.

> **What changes:** initially, all agents have batteries, grid import/export and signed bilateral trades. No EV, heat pump, IEEE33 feeder or trade-absolute auxiliaries. They are added *after* the basic ladder works. The original toy differs from its predecessor because this experiment uses ε=0.01 kW rather than a strict shared equality.


In [3]:
# Cell 1 — Imports and package provenance (execute once)
import sys, os, json, time, pickle, signal, subprocess, importlib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
from IPython.display import display

from dynect import DynLQGameTV, MpAVI
from dynect.solvers.gne import GNESetSolver
from p2p.setup import GameBundle, to_static_gnep

p2p_root = Path(importlib.import_module("p2p.setup").__file__).resolve().parents[1]
dynect_root = Path(importlib.import_module("dynect").__file__).resolve().parents[1]
print("Python:", sys.executable)
print("P2P adapter:", p2p_root)
print("DyNECT:", dynect_root)


Python: /opt/anaconda3/envs/p2p/bin/python
P2P adapter: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect
DyNECT: /Users/catharinahente/Documents/Github/dynect_py_p2p


## 1 · Build a clean LQ P2P game

This deliberately follows the control conventions of the original two-player toy: charge, discharge, grid import, grid export and one signed peer-power variable per trading neighbor. It preserves battery dynamics and individual power balance. The four-prosumer version uses the same cyclic trade graph as your study.

Optional `abs_aux=True` then adds trade absolute-value epigraph controls; `extra_fixed=3` simulates additional controls forced to zero. These are **controlled implementation stress tests**, not substitutes for the real device model.

In [4]:
# Cell 2 — Self-contained time-varying game builder
TOY_LOAD = np.array([[1.2, 1.5, 1.1, 1.7],
                     [1.3, 1.2, 1.8, 1.7]])
TOY_PV   = np.array([[2.2, 2.5, 0.3, 0.0],
                     [0.0, 0.2, 0.2, 0.1]])
BUY = np.array([0.28, 0.34, 0.39, 0.30])
SELL = np.array([0.07, 0.07, 0.09, 0.08])
P2P = np.array([0.17, 0.20, 0.23, 0.18])

def build_toy_bundle(N=2, T=4, *, eps=0.01, dt=1.0,
                     abs_aux=False, extra_fixed=0, edge_mode='ring'):
    assert N in (2, 4) and T >= 1 and eps >= 0 and extra_fixed >= 0
    assert edge_mode in ('ring','pairs')
    edges = [(0, 1)] if N == 2 else ([(0, 1), (2, 3)] if edge_mode == 'pairs' else [(0, 1), (1, 2), (2, 3), (0, 3)])
    peers = [[j for e in edges for j in e if i in e and j != i] for i in range(N)]
    peers = [sorted(set(p)) for p in peers]

    layouts = []
    for i in range(N):
        ix = {name:k for k,name in enumerate(['charge','discharge','buy','sell'])}
        for j in peers[i]:
            ix[f'trade_{j}'] = len(ix)
            if abs_aux:
                ix[f'trade_abs_{j}'] = len(ix)
        for k in range(extra_fixed):
            ix[f'fixed_{k}'] = len(ix)
        layouts.append(ix)
    nu = [len(x) for x in layouts]
    x0 = np.full(N, 2.0)
    A, B, R, r, C_loc, b_loc, C_u, b_u, C_x, b_x = ([] for _ in range(10))

    for t in range(T):
        Bt = []; Rt = []; rt = []; Clt = []; blt = []
        for i in range(N):
            ix = layouts[i]; ni = nu[i]
            Bi = np.zeros((N, ni))
            Bi[i, ix['charge']] = dt * 0.95
            Bi[i, ix['discharge']] = -dt / 0.95
            Bt.append(Bi)

            # Match the original toy's strictly positive own-control curvature.
            diagonal = np.full(ni, 0.004)
            diagonal[ix['charge']] = 0.04
            diagonal[ix['discharge']] = 0.04
            diagonal[ix['buy']] = 0.002
            diagonal[ix['sell']] = 0.002
            for j in peers[i]:
                diagonal[ix[f'trade_{j}']] = 0.004
                if abs_aux: diagonal[ix[f'trade_abs_{j}']] = 0.002
            Rt.append([np.diag(diagonal) if i==j else np.zeros((ni, nu[j]))
                       for j in range(N)])

            ri = np.zeros(ni)
            ri[ix['charge']] = dt * 0.015
            ri[ix['discharge']] = dt * 0.015
            ri[ix['buy']] = dt * BUY[t % 4]
            ri[ix['sell']] = -dt * SELL[t % 4]
            for j in peers[i]:
                ri[ix[f'trade_{j}']] = dt * P2P[t % 4]
                if abs_aux:
                    ri[ix[f'trade_abs_{j}']] = dt * 0.001
            rt.append(ri)

            # Local input bounds, equality-pair power balance, and optional epigraph.
            maxvals = np.full(ni, 3.0)
            maxvals[ix['charge']] = 1.5
            maxvals[ix['discharge']] = 1.5
            maxvals[ix['buy']] = 5.0
            maxvals[ix['sell']] = 5.0
            lower = np.zeros(ni)
            for j in peers[i]: lower[ix[f'trade_{j}']] = -3.0
            for k in range(extra_fixed):
                maxvals[ix[f'fixed_{k}']] = 0.0
                lower[ix[f'fixed_{k}']] = 0.0
            C = [*np.eye(ni), *(-np.eye(ni))]
            b = [*maxvals, *(-lower)]
            for j in peers[i]:
                if abs_aux:
                    row = np.zeros(ni)
                    row[ix[f'trade_{j}']] = 1
                    row[ix[f'trade_abs_{j}']] = -1
                    row2 = -row.copy()
                    row2[ix[f'trade_abs_{j}']] = -1
                    C.extend([row, row2])
                    b.extend([0., 0.])

            bal = np.zeros(ni)
            bal[ix['charge']] = -1
            bal[ix['discharge']] = 1
            bal[ix['buy']] = 1
            bal[ix['sell']] = -1
            for j in peers[i]: bal[ix[f'trade_{j}']] = 1
            if N == 2 and T == 4:
                demand = TOY_LOAD[i,t] - TOY_PV[i,t]
            else:
                demand = (1.2 + 0.16*i + 0.20*np.cos(2*np.pi*(t+.3)/max(T,2))
                          - (1.0 if i % 2 == 0 else 0.18)
                          * max(0, np.sin(2*np.pi*(t+.5)/max(T,2))))
            C.extend([bal, -bal]); b.extend([demand, -demand])
            Clt.append(np.asarray(C,float))
            blt.append(np.asarray(b,float))

        # Shared P2P reciprocity: sum of directed trades on each edge ≈ 0.
        Cu_t = [np.zeros((2*len(edges),nu[i])) for i in range(N)]
        for k,(i,j) in enumerate(edges):
            Cu_t[i][2*k,   layouts[i][f'trade_{j}']] =  1
            Cu_t[j][2*k,   layouts[j][f'trade_{i}']] =  1
            Cu_t[i][2*k+1, layouts[i][f'trade_{j}']] = -1
            Cu_t[j][2*k+1, layouts[j][f'trade_{i}']] = -1
        A.append(np.eye(N)); B.append(Bt); R.append(Rt); r.append(rt)
        C_loc.append(Clt); b_loc.append(blt)
        C_u.append(Cu_t); b_u.append(np.full(2*len(edges),eps))
        C_x.append(np.vstack([np.eye(N),-np.eye(N)]))
        b_x.append(np.r_[np.full(N,4.), np.full(N,-.5)])

    Q = [[1e-4*np.eye(N) for i in range(N)] for _ in range(T-1)]
    P = []; p = []
    for i in range(N):
        Pi = 1e-4 * np.eye(N); Pi[i,i] += .10
        pi = np.zeros(N); pi[i] = -.10 * 2.0
        P.append(Pi); p.append(pi)

    game = DynLQGameTV(A=A,B=B,Q=Q,R=R,P=P,p=p,r=r,
                       C_x=C_x,b_x=b_x,C_loc=C_loc,b_loc=b_loc,
                       C_u=C_u,b_u=b_u)
    return GameBundle(game=game,x0=x0,data={},grid=None,
                      state_names=[f'SOC{i}' for i in range(N)],
                      state_index={},controls=layouts,peers=peers,
                      shared_labels=[f'edge_{i}_{j}_{sgn}' for i,j in edges
                                     for sgn in ('plus','minus')],
                      choices={'toy':True,'eps':eps,'abs_aux':abs_aux,
                               'extra_fixed':extra_fixed, 'edge_mode':edge_mode})

# Verify that our baseline has the dimensions of the earlier toy.
toy = build_toy_bundle(N=2,T=4)
_, avi_toy, static_toy = to_static_gnep(toy)
print('Toy:', 'variables',sum(static_toy.n), 'shared',static_toy.m_sh,
      'beta',(static_toy.N-1)*static_toy.m_sh,
      'condensed constraints',len(avi_toy.b))
assert sum(static_toy.n)==40 and static_toy.m_sh==8


Toy: variables 40 shared 8 beta 8 condensed constraints 120


## 2 · Ladder: advance only after a stage actually explores a region

This deliberately separates **player count**, **trade-graph connectivity**, **time horizon**, **extra trading auxiliaries** and **fixed-zero controls**. Only compare one changed assumption at a time; time-varying load is synthetic at this stage.


In [5]:
# Cell 3 — Choose a ladder stage. Start at 1, the N=2 / T=4 working scale.
STAGES = [
    ('0 · 2P / 1 step',              dict(N=2,T=1)),
    ('1 · 2P / 4 steps — toy scale', dict(N=2,T=4)),
    ('2 · 2P / 6 steps',            dict(N=2,T=6)),
    ('3 · 4P / 4 steps — two disconnected pairs', dict(N=4,T=4,edge_mode='pairs')),
    ('4 · 4P / 4 steps — cyclic trading', dict(N=4,T=4)),
    ('5 · 4P / 1 step — ring',      dict(N=4,T=1)),
    ('6 · 4P / 2 steps — ring',     dict(N=4,T=2)),
    ('7 · 4P / 6 steps — ring',     dict(N=4,T=6)),
    ('8 · Stage 7 + |trades| auxiliaries', dict(N=4,T=6,abs_aux=True)),
    ('9 · Stage 8 + 3 fixed-zero controls per player',
     dict(N=4,T=6,abs_aux=True,extra_fixed=3)),
]
SELECT_STAGE = 1
stage_name, settings = STAGES[SELECT_STAGE]
case = build_toy_bundle(**settings)
mp, avi, gnep = to_static_gnep(case)
nbeta = (gnep.N-1)*gnep.m_sh
print('Stage:',stage_name)
print('N, T =',case.N,case.T,'Controls/player=',case.game.nu)
print('Decisions =',sum(gnep.n),'local inequalities =',sum(len(b) for b in gnep.b_loc))
print('AVI inequalities =',len(avi.b),'shared inequalities =',gnep.m_sh,'beta =',nbeta)


Stage: 1 · 2P / 4 steps — toy scale
N, T = 2 4 Controls/player= [5, 5]
Decisions = 40 local inequalities = 112
AVI inequalities = 120 shared inequalities = 8 beta = 8


## 3 · Inspect the actual transformed problem **before Julia**

`GNESetSolver` first calls `MpAVI.nabetani(...)` in Python. The resulting dimensions—not the original AVI's 670 inequalities—are the dimensions that matter to the Julia region algorithm. This cell measures that conversion and checks finiteness, curvature, constraint rows and duplicated/opposite rows. It **does not launch enumeration**.


In [6]:
# Cell 4 — Nabetani preprocessing diagnostic (no Julia)
BETA = 5.0
start = time.perf_counter()
mpp = MpAVI.nabetani(gnep,
    beta_lb=np.full(nbeta, -BETA),beta_ub=np.full(nbeta,BETA))
print('Nabetani construction time [s]:',round(time.perf_counter()-start,3))
for name in ('H','F','f','A','B','b','C','d','lb','ub'):
    a=np.asarray(getattr(mpp,name),float)
    print(f'{name:>3}: shape={str(a.shape):>15}  min={np.min(a) if a.size else np.nan:.3g} '
          f'max={np.max(a) if a.size else np.nan:.3g} finite={np.isfinite(a).all()}')
Hm = np.asarray(mpp.H,float)
if Hm.shape[0]==Hm.shape[1]:
    ev=np.linalg.eigvalsh((Hm+Hm.T)/2)
    print('Transformed symmetric Jacobian eig range:',float(ev[0]),float(ev[-1]))
Av=np.asarray(mpp.A,float)
if Av.ndim==2:
    print('Transformed constraint rows:',Av.shape[0],
          'zero rows:',int(np.count_nonzero(np.linalg.norm(Av,axis=1)<1e-12)))


Nabetani construction time [s]: 0.013
  H: shape=       (40, 40)  min=-0.1 max=0.151 finite=True
  F: shape=        (40, 8)  min=0 max=0 finite=True
  f: shape=          (40,)  min=-0.09 max=0.39 finite=True
  A: shape=      (128, 40)  min=-1.05 max=1.05 finite=True
  B: shape=       (128, 8)  min=-1 max=1 finite=True
  b: shape=         (128,)  min=-1.7 max=5 finite=True
  C: shape=         (0, 8)  min=nan max=nan finite=True
  d: shape=           (0,)  min=nan max=nan finite=True
 lb: shape=           (8,)  min=-5 max=-5 finite=True
 ub: shape=           (8,)  min=5 max=5 finite=True
Transformed symmetric Jacobian eig range: 0.0019999999999999415 0.8457214598262084
Transformed constraint rows: 128 zero rows: 0


## 4 · Direct Julia connection check

This tests Julia through **the same Python environment**, independently of region enumeration. `2` establishes a working `juliacall` bridge, **not** a healthy ParametricDAQP optimization.

In [7]:
# Cell 5 — Julia smoke test (no enumeration)
from juliacall import Main as jl
print('Julia 1 + 1 =',jl.seval('1+1'))
print('Julia version:',jl.seval('string(VERSION)'))
assert jl.seval('1+1')==2


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Julia 1 + 1 = 2
Julia version: 1.13.4


## 5 · Run ONE stage with timeout-only protection and live logs

The worker uses the installed `GNESetSolver` **without overrides**. The worker independently prints Nabetani construction time before `GNESetSolver` so that silence can be attributed to Julia rather than to construction of the parametric game. It also performs the Julia smoke test. The notebook displays new log lines and a heartbeat every 10 seconds.

Select a new `SELECT_STAGE`, run Cells 3–4, then this cell. **Don't immediately proceed to harder stages if the toy-size stage fails in this subprocess**; that suggests a mismatch with the previously working environment or toy formulation.


In [13]:
# Cell 6 — One protected GNE attempt
REGION_LIMIT = 250
TIMEOUT_SECONDS = 600
VERBOSE = 2
RUN_ENUMERATION = True

if RUN_ENUMERATION:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(p2p_root),str(dynect_root),
                                        env.get('PYTHONPATH','')])
    env['PYTHONUNBUFFERED'] = '1'
    run_dir = p2p_root / '.gne_experiments' / datetime.now().strftime('bottom_up_%Y%m%d_%H%M%S_%f')
    run_dir.mkdir(parents=True,exist_ok=True)
    input_file, worker_file = run_dir/'case.pkl', run_dir/'worker.py'
    log_file, result_file = run_dir/'solver.log', run_dir/'result.json'
    with input_file.open('wb') as f: pickle.dump(case,f)

    worker_file.write_text(r"""
import json, pickle, sys, time, traceback
from pathlib import Path
import numpy as np
from juliacall import Main as jl
from p2p.setup import to_static_gnep
from dynect import MpAVI
from dynect.solvers.gne import GNESetSolver

def log(*a): print(time.strftime('%H:%M:%S'),*a,flush=True)
try:
    inpath, outpath = map(Path,sys.argv[1:3])
    beta=float(sys.argv[3]); limit=int(sys.argv[4]); verbose=int(sys.argv[5])
    log('Julia smoke test:',jl.seval('1+1'),'version',jl.seval('string(VERSION)'))
    with inpath.open('rb') as f: case=pickle.load(f)
    start=time.perf_counter(); _,avi,gnep=to_static_gnep(case)
    log('StaticGNEP ready in',round(time.perf_counter()-start,3),'s',
        'decisions',sum(gnep.n),'m_sh',gnep.m_sh)
    nb=(gnep.N-1)*gnep.m_sh
    start=time.perf_counter()
    p=MpAVI.nabetani(gnep,beta_lb=np.full(nb,-beta),beta_ub=np.full(nb,beta))
    log('Nabetani ready in',round(time.perf_counter()-start,3),'s',
        'H',p.H.shape,'A',p.A.shape,'C',p.C.shape)
    log('ENTER GNESetSolver (Julia region search)')
    start=time.perf_counter()
    solver=GNESetSolver(gnep,beta_lb=np.full(nb,-beta),beta_ub=np.full(nb,beta),
                        verbose=verbose,region_limit=limit)
    summary={'status':str(solver.status),'gne_regions':len(solver.gne_set.regions),
             'elapsed_solver_s':round(time.perf_counter()-start,2),
             'decisions':sum(gnep.n),'beta':nb}
    outpath.write_text(json.dumps(summary,indent=2))
    log('DONE:',summary)
except Exception as e:
    log('ERROR:',repr(e));traceback.print_exc()
    sys.exit(1)
""")

    start=time.monotonic(); offset=0; last_print=start
    print('Trial:',stage_name,'limit:',REGION_LIMIT,'timeout:',TIMEOUT_SECONDS,'s',flush=True)
    print('Run folder:',run_dir,flush=True)
    timed_out=False
    with log_file.open('w') as log:
        proc=subprocess.Popen([sys.executable,'-u',str(worker_file),str(input_file),
                               str(result_file),str(BETA),str(REGION_LIMIT),str(VERBOSE)],
                              stdout=log,stderr=subprocess.STDOUT,cwd=p2p_root,
                              env=env,start_new_session=True)
        while proc.poll() is None:
            time.sleep(0.5); elapsed=time.monotonic()-start
            if elapsed>TIMEOUT_SECONDS:
                timed_out=True
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                break
            if time.monotonic()-last_print>10:
                with log_file.open(errors='replace') as f:
                    f.seek(offset); output=f.read(); offset=f.tell()
                print(f'[{elapsed:5.1f}s] running',flush=True)
                if output: print(output[-2500:],flush=True)
                last_print=time.monotonic()
        proc.wait()
    print('Exit code:',proc.returncode,'timed out:',timed_out,
          'elapsed [s]:',round(time.monotonic()-start,2))
    print('Result:',result_file.read_text() if result_file.exists() else 'No completed result')
    print('Final log tail:\n',log_file.read_text(errors='replace')[-5000:])


Trial: 1 · 2P / 4 steps — toy scale limit: 250 timeout: 600 s
Run folder: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/.gne_experiments/bottom_up_20261125_112014_142780
[ 10.3s] running
[ 20.4s] running
11:20:28 Julia smoke test: 2 version 1.13.4
11:20:28 StaticGNEP ready in 0.014 s decisions 40 m_sh 8
11:20:28 Nabetani ready in 0.001 s H (40, 40) A (128, 40) C (0, 8)
11:20:28 ENTER GNESetSolver (Julia region search)

[ 30.5s] running
[ 40.6s] running
[ 50.7s] running
[ 60.8s] running
[ 70.9s] running
[ 81.1s] running
[ 91.3s] running
[101.5s] running
[111.7s] running
Exit code: 0 timed out: False elapsed [s]: 120.34
Result: {
  "status": "Status.REGION_LIMIT",
  "gne_regions": 0,
  "elapsed_solver_s": 105.56,
  "decisions": 40,
  "beta": 8
}
Final log tail:
 11:20:28 Julia smoke test: 2 version 1.13.4
11:20:28 StaticGNEP ready in 0.014 s decisions 40 m_sh 8
11:20:28 Nabetani ready in 0.001 s H (40, 40) A (128, 40) C (0, 8)
11:20:28 ENTER GNESetSolv

## 6 · How to interpret results

- **Stage 1 works but Stage 3 fails**: four players with two independent trading pairs already burden the global Nabetani parameterization; the issue need not be the battery model.
- **Stage 3 works but Stage 4 fails**: cyclic trading topology / additional shared constraints are implicated.
- **Stage 7 works but Stage 8 fails**: trade-absolute epigraph auxiliary controls/local inequalities are implicated.
- **Stage 8 works but Stage 9 fails**: fixed-zero variables/degenerate inequality pairs are implicated.
- **Stage 1 fails, but the supervisor's original sample works**: check package revision, different toy dynamics, initial β bounds, and solver wrapper rather than simplifying real-world data.
- **Julia smoke test works but no regions appear**: the bridge works; initial-region search / preprocessing / numerical degeneracy is implicated. No direct evidence that an active region was ever found.
- **Region limit reached**: a partial set may be returned; it is **not** a certificate of all GNEs or of globally optimal fairness.

A feasible vGNE of the original condensed AVI is **not** a ready-made `AS0` for the Nabetani-transformed mpAVI: constraints are transformed/reordered and additional β variables enter. An initial-region warm start needs a valid active set in the transformed problem with a compatible parameter point.

Once the first failure is isolated, the next ladder should add **real** demand/PV/tariffs, then device-specific bounds, then EV/HP constraints, and finally feeder constraints, while keeping the winning (N,T) size fixed.

# Warm Start

In [14]:
from p2p_gne_as0_warmstart import prepare_as0, run_with_as0

# Use exactly the same Stage 1 model as before.
# Only rebuild if you've overwritten the variable 'case'.
case = build_toy_bundle(N=2, T=4, eps=0.01)

BETA = 5.0  # Same beta bounds as previous experiment

warm = prepare_as0(
    case,
    beta=BETA,
    tol=2e-6,
)

print("\n=== WARM START READY ===")
print("AS0 size:", len(warm["as0"]))
print("AS0 indices:", warm["as0"])
print("Maximum primal violation:", warm["primal_violation"])
print("KKT stationarity residual:", warm["stationarity"])
print("Active-set rank:", warm["rank"])

Transformed dimensions: n=40, inequalities=128, θ=8, β=8
Symmetric H error=1.388e-17; min eig(Hsym)=2.000e-03
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806500
Academic license 2806500 - for non-commercial use only - registered to ch___@ethz.ch
Fixed-θ QP solved; max primal violation: 4.44e-16
Tight rows: 50 positive-dual rows: 37 rank: 37
Stationarity residual: 5.14e-10 complementarity residual: 1.91e-11

=== WARM START READY ===
AS0 size: 37
AS0 indices: [14, 15, 19, 26, 27, 31, 37, 39, 40, 43, 45, 49, 52, 55, 61, 63, 69, 70, 72, 75, 81, 84, 87, 93, 96, 99, 105, 108, 111, 113, 115, 117, 118, 120, 122, 124, 126]
Maximum primal violation: 4.440892098500626e-16
KKT stationarity residual: 5.140261549740899e-10
Active-set rank: 37


In [15]:
print("=== WARM-STARTED GNE ENUMERATION ===")

run_with_as0(
    case,
    warm,
    beta=5.0,
    region_limit=250,
    timeout_seconds=600,
    verbose=2,
)

=== WARM-STARTED GNE ENUMERATION ===
Experiment: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/.gne_experiments/AS0_20261125_112751_511605
[10 s] worker active
11:27:59 Nabetani ready: decisions= 40 rows= 128 beta= 8 AS0 size= 37
11:27:59 ENTER mpsolve, warm start AS0, region_limit= 250

[20 s] worker active
[31 s] worker active
[41 s] worker active
[52 s] worker active
[62 s] worker active
[72 s] worker active
[82 s] worker active
[92 s] worker active
[103 s] worker active
[113 s] worker active
[123 s] worker active
[134 s] worker active
[144 s] worker active
[154 s] worker active
[164 s] worker active
[175 s] worker active
[185 s] worker active
[195 s] worker active
[205 s] worker active
[216 s] worker active
[228 s] worker active
[239 s] worker active
[250 s] worker active


KeyboardInterrupt: 